In [1]:
import os
import shutil
import cv2
import numpy as np
import mediapipe as mp
from tqdm import tqdm


# ============================================================
# PATHS
# ============================================================

BASE_DIR = r"D:\aaa EAAI Major Revision\dynamic\dynamic_dataset"

INPUT_DIR = os.path.join(BASE_DIR, "merged")

OUTPUT_NAMES = {
    "train": "train_csv",
    "test": "test_csv",
    "validation": "validation_csv"
}

SPLITS = ["train", "test", "validation"]

TARGET_FRAMES = 120


# ============================================================
# MEDIAPIPE
# ============================================================

mp_hands = mp.solutions.hands
mp_pose = mp.solutions.pose


# ============================================================
# EXTRACT 130 FEATURES
# ============================================================

def extract_features(frame, hands, pose):

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    hand_results = hands.process(rgb_frame)
    pose_results = pose.process(rgb_frame)

    # --------------------------------------------------------
    # LEFT + RIGHT HAND
    # 21 landmarks × 3 = 63 each
    # --------------------------------------------------------

    left_hand = np.zeros(63, dtype=np.float32)
    right_hand = np.zeros(63, dtype=np.float32)

    if hand_results.multi_hand_landmarks:

        for hand_landmarks, handedness in zip(
            hand_results.multi_hand_landmarks,
            hand_results.multi_handedness
        ):

            label = handedness.classification[0].label

            landmarks = []

            for landmark in hand_landmarks.landmark:
                landmarks.extend([
                    landmark.x,
                    landmark.y,
                    landmark.z
                ])

            landmarks = np.array(
                landmarks,
                dtype=np.float32
            )

            if label == "Left":
                left_hand = landmarks

            elif label == "Right":
                right_hand = landmarks

    # --------------------------------------------------------
    # POSE DISTANCES
    #
    # 0  = Nose
    # 13 = Left Elbow
    # 14 = Right Elbow
    # 15 = Left Wrist
    # 16 = Right Wrist
    # --------------------------------------------------------

    distances = np.zeros(4, dtype=np.float32)

    if pose_results.pose_landmarks:

        p = pose_results.pose_landmarks.landmark

        nose = np.array([
            p[0].x,
            p[0].y,
            p[0].z
        ])

        left_wrist = np.array([
            p[15].x,
            p[15].y,
            p[15].z
        ])

        right_wrist = np.array([
            p[16].x,
            p[16].y,
            p[16].z
        ])

        left_elbow = np.array([
            p[13].x,
            p[13].y,
            p[13].z
        ])

        right_elbow = np.array([
            p[14].x,
            p[14].y,
            p[14].z
        ])

        distances[0] = np.linalg.norm(
            nose - left_wrist
        )

        distances[1] = np.linalg.norm(
            nose - right_wrist
        )

        distances[2] = np.linalg.norm(
            nose - left_elbow
        )

        distances[3] = np.linalg.norm(
            nose - right_elbow
        )

    # --------------------------------------------------------
    # TOTAL = 63 + 63 + 4 = 130
    # --------------------------------------------------------

    features = np.concatenate([
        left_hand,
        right_hand,
        distances
    ])

    assert features.shape == (130,)

    return features


# ============================================================
# CHECK WHETHER A CLASS IS COMPLETE
# ============================================================

def is_class_complete(class_input_folder, class_output_folder):

    # Output class folder must exist
    if not os.path.isdir(class_output_folder):
        return False

    # Get input videos
    videos = sorted([
        f for f in os.listdir(class_input_folder)
        if f.lower().endswith(".avi")
    ])

    # There must be at least one video
    if len(videos) == 0:
        return False

    # --------------------------------------------------------
    # Every video must have its own output folder
    # containing exactly 120 .npy files
    # --------------------------------------------------------

    for video_name in videos:

        video_stem = os.path.splitext(video_name)[0]

        video_output_folder = os.path.join(
            class_output_folder,
            video_stem
        )

        if not os.path.isdir(video_output_folder):
            return False

        npy_files = [
            f for f in os.listdir(video_output_folder)
            if f.lower().endswith(".npy")
        ]

        # Must have exactly 120 frames
        if len(npy_files) != TARGET_FRAMES:
            return False

        # Make sure 001.npy ... 120.npy all exist
        for frame_number in range(1, TARGET_FRAMES + 1):

            expected_file = os.path.join(
                video_output_folder,
                f"{frame_number:03d}.npy"
            )

            if not os.path.isfile(expected_file):
                return False

    return True


# ============================================================
# PROCESS ONE VIDEO
# ============================================================

def process_video(video_path, output_video_folder):

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print(f"ERROR: Could not open {video_path}")
        return False

    total_frames = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    if total_frames < TARGET_FRAMES:

        print(
            f"ERROR: {video_path} has only "
            f"{total_frames} frames."
        )

        cap.release()
        return False

    os.makedirs(
        output_video_folder,
        exist_ok=True
    )

    # --------------------------------------------------------
    # MediaPipe
    # --------------------------------------------------------

    with mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        model_complexity=1,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as hands, mp_pose.Pose(
        static_image_mode=False,
        model_complexity=1,
        smooth_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as pose:

        for frame_index in range(TARGET_FRAMES):

            cap.set(
                cv2.CAP_PROP_POS_FRAMES,
                frame_index
            )

            success, frame = cap.read()

            if not success:

                print(
                    f"ERROR reading frame "
                    f"{frame_index + 1}: {video_path}"
                )

                cap.release()
                return False

            features = extract_features(
                frame,
                hands,
                pose
            )

            output_path = os.path.join(
                output_video_folder,
                f"{frame_index + 1:03d}.npy"
            )

            np.save(
                output_path,
                features
            )

    cap.release()

    return True


# ============================================================
# PROCESS ONE CLASS
# ============================================================

def process_class(
    class_input_folder,
    class_output_folder
):

    videos = sorted([
        f for f in os.listdir(class_input_folder)
        if f.lower().endswith(".avi")
    ])

    print()
    print("=" * 75)
    print(
        f"PROCESSING CLASS: "
        f"{os.path.basename(class_input_folder)}"
    )
    print(f"Videos: {len(videos)}")
    print("=" * 75)

    for video_name in tqdm(
        videos,
        desc=os.path.basename(class_input_folder)
    ):

        video_path = os.path.join(
            class_input_folder,
            video_name
        )

        video_stem = os.path.splitext(
            video_name
        )[0]

        output_video_folder = os.path.join(
            class_output_folder,
            video_stem
        )

        success = process_video(
            video_path,
            output_video_folder
        )

        if not success:
            print(
                f"\nERROR processing video: "
                f"{video_name}"
            )
            print(
                "The class will be detected as "
                "INCOMPLETE on the next check."
            )


# ============================================================
# PROCESS SPLIT
# ============================================================

def process_split(split):

    input_split_dir = os.path.join(
        INPUT_DIR,
        split
    )

    output_split_dir = os.path.join(
        BASE_DIR,
        OUTPUT_NAMES[split]
    )

    os.makedirs(
        output_split_dir,
        exist_ok=True
    )

    classes = sorted([
        d for d in os.listdir(input_split_dir)
        if os.path.isdir(
            os.path.join(input_split_dir, d)
        )
    ])

    print()
    print("#" * 80)
    print(f"STARTING SPLIT: {split}")
    print(f"Total classes: {len(classes)}")
    print("#" * 80)

    for class_name in classes:

        class_input_folder = os.path.join(
            input_split_dir,
            class_name
        )

        class_output_folder = os.path.join(
            output_split_dir,
            class_name
        )

        # ====================================================
        # CHECK EXISTING CLASS
        # ====================================================

        if os.path.exists(class_output_folder):

            if is_class_complete(
                class_input_folder,
                class_output_folder
            ):

                # ------------------------------------------------
                # COMPLETE → SKIP
                # ------------------------------------------------

                print(
                    f"\n[COMPLETE - SKIP] {class_name}"
                )

                continue

            else:

                # ------------------------------------------------
                # INCOMPLETE → DELETE EVERYTHING
                # ------------------------------------------------

                print()
                print(
                    f"[INCOMPLETE] {class_name}"
                )
                print(
                    "Deleting incomplete class folder..."
                )

                shutil.rmtree(
                    class_output_folder
                )

                print(
                    "Deleted. Restarting this class "
                    "from the beginning."
                )

        # ====================================================
        # CLASS DOES NOT EXIST OR WAS INCOMPLETE
        # ====================================================

        process_class(
            class_input_folder,
            class_output_folder
        )

        # ====================================================
        # VERIFY AFTER PROCESSING
        # ====================================================

        if is_class_complete(
            class_input_folder,
            class_output_folder
        ):

            print(
                f"\n[COMPLETED] {class_name}"
            )

        else:

            print()
            print(
                f"[WARNING] {class_name} "
                f"is still incomplete!"
            )

            print(
                "It will be automatically "
                "reprocessed if the program is restarted."
            )

    print()
    print("#" * 80)
    print(f"FINISHED SPLIT: {split}")
    print("#" * 80)


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    for split in SPLITS:

        process_split(split)

    print()
    print("=" * 80)
    print("ALL PROCESSING COMPLETED")
    print("=" * 80)


################################################################################
STARTING SPLIT: train
Total classes: 40
################################################################################

[COMPLETE - SKIP] Good to See You

[COMPLETE - SKIP] Leave

[COMPLETE - SKIP] Please

[COMPLETE - SKIP] What Time is It

[COMPLETE - SKIP] Where Do You Live

[COMPLETE - SKIP] abar dekha hbe

[COMPLETE - SKIP] ami dukkhito

[COMPLETE - SKIP] ami valo achi

[COMPLETE - SKIP] apnake amar valo legeche

[COMPLETE - SKIP] apnake kivabe sahajjo korte pari

[COMPLETE - SKIP] apnar nam ki

[COMPLETE - SKIP] apni kemon achen

[COMPLETE - SKIP] apni ki kaj koren

[COMPLETE - SKIP] apni valo thakben

[COMPLETE - SKIP] asha

[COMPLETE - SKIP] baba

[COMPLETE - SKIP] boi

[COMPLETE - SKIP] bon

[COMPLETE - SKIP] college

[COMPLETE - SKIP] computer

[COMPLETE - SKIP] dekha

[COMPLETE - SKIP] dhonnobad

[COMPLETE - SKIP] fan

[COMPLETE - SKIP] ghumano

[COMPLETE - SKIP] hata

[COMPLETE - SKIP] hello


abar dekha hbe: 100%|██████████| 31/31 [06:07<00:00, 11.86s/it]



[COMPLETED] abar dekha hbe

PROCESSING CLASS: ami dukkhito
Videos: 31


ami dukkhito: 100%|██████████| 31/31 [05:54<00:00, 11.43s/it]



[COMPLETED] ami dukkhito

PROCESSING CLASS: ami valo achi
Videos: 28


...


[COMPLETED] shuvo sokal

PROCESSING CLASS: stri
Videos: 29


stri: 100%|██████████| 29/29 [04:44<00:00,  9.82s/it]



[COMPLETED] stri

PROCESSING CLASS: vai
Videos: 29


vai: 100%|██████████| 29/29 [04:38<00:00,  9.60s/it]


[COMPLETED] vai

################################################################################
FINISHED SPLIT: validation
################################################################################

ALL PROCESSING COMPLETED
